In [1]:
# cell 1

# Add src directory to path
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)


In [2]:
# cell 2

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

from models import get_models
from evaluation import evaluate_model
from noise import add_symmetric_noise, add_asymmetric_noise

np.random.seed(42)

In [3]:
# cell 3

columns = [
"age","workclass","fnlwgt","education","education-num",
"marital-status","occupation","relationship","race","sex",
"capital-gain","capital-loss","hours-per-week","native-country","income"
]

df_train = pd.read_csv("../data/adult/adult.data",
                       names=columns,
                       na_values=" ?",
                       skipinitialspace=True)

df_test = pd.read_csv("../data/adult/adult.test",
                      names=columns,
                      na_values=" ?",
                      skiprows=1,
                      skipinitialspace=True)

df = pd.concat([df_train, df_test], ignore_index=True)

df = df.dropna()

# fix test label format
df["income"] = df["income"].str.replace(".", "", regex=False)

# Explicit categorical columns
categorical_cols = [
"workclass",
"education",
"marital-status",
"occupation",
"relationship",
"race",
"sex",
"native-country",
"income"
]

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

# convert to numpy
X = df.drop("income", axis=1).values
y = df["income"].values

print("Dataset shape:", X.shape)

Dataset shape: (48842, 14)


In [4]:
# cell 4

noise_levels = [0.0, 0.1, 0.2, 0.3]
noise_types = ["symmetric", "asymmetric"]

n_splits = 5

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

models = get_models()

In [5]:
# cell 5

results = []

for noise_type in noise_types:
    for noise_rate in noise_levels:
        
        fold_idx = 1
        
        for train_index, test_index in skf.split(X, y):
            
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]
            
            # Apply noise ONLY to training labels
            if noise_rate == 0.0:
                y_train_noisy = y_train.copy()
            else:
                if noise_type == "symmetric":
                    y_train_noisy = add_symmetric_noise(y_train, noise_rate)
                elif noise_type == "asymmetric":
                    y_train_noisy = add_asymmetric_noise(y_train, noise_rate)
            
            # Train all models
            for model_name, model in models.items():
                
                acc, f1 = evaluate_model(
                    model,
                    X_train,
                    y_train_noisy,
                    X_test,
                    y_test
                )
                
                results.append({
                    "Noise_Type": noise_type,
                    "Noise_Rate": noise_rate,
                    "Fold": fold_idx,
                    "Model": model_name,
                    "Accuracy": acc,
                    "F1": f1
                })
            
            fold_idx += 1

results_df = pd.DataFrame(results)
results_df.head()

[LightGBM] [Info] Number of positive: 9349, number of negative: 29724
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002285 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 721
[LightGBM] [Info] Number of data points in the train set: 39073, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.239270 -> initscore=-1.156685
[LightGBM] [Info] Start training from score -1.156685


C:\Users\Garima\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 9349, number of negative: 29724
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002474 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 721
[LightGBM] [Info] Number of data points in the train set: 39073, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.239270 -> initscore=-1.156685
[LightGBM] [Info] Start training from score -1.156685


C:\Users\Garima\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 9350, number of negative: 29724
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002442 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 717
[LightGBM] [Info] Number of data points in the train set: 39074, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.239290 -> initscore=-1.156578
[LightGBM] [Info] Start training from score -1.156578


...

[LightGBM] [Info] Number of positive: 6545, number of negative: 32529
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001831 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 723
[LightGBM] [Info] Number of data points in the train set: 39074, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.167503 -> initscore=-1.603431
[LightGBM] [Info] Start training from score -1.603431


C:\Users\Garima\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 6545, number of negative: 32529
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002932 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 712
[LightGBM] [Info] Number of data points in the train set: 39074, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.167503 -> initscore=-1.603431
[LightGBM] [Info] Start training from score -1.603431


C:\Users\Garima\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Noise_Type,Noise_Rate,Fold,Model,Accuracy,F1
0,symmetric,0.0,1,Logistic Regression,0.825571,0.553927
1,symmetric,0.0,1,Decision Tree,0.811547,0.617176
2,symmetric,0.0,1,Random Forest,0.859351,0.681354
3,symmetric,0.0,1,SVM,0.848398,0.635670
4,symmetric,0.0,1,KNN,0.828335,0.618950


In [6]:
# cell 6

len(results_df)

240

In [7]:
# cell 7

results_df[
    (results_df["Noise_Rate"] == 0.0)
].groupby(["Noise_Type", "Model"])["Accuracy"].mean()

Noise_Type  Model              
asymmetric  Decision Tree          0.812129
            KNN                    0.829163
            LightGBM               0.874166
            Logistic Regression    0.824884
            Random Forest          0.856824
            SVM                    0.849494
symmetric   Decision Tree          0.814094
            KNN                    0.829163
            LightGBM               0.874166
            Logistic Regression    0.824884
            Random Forest          0.858892
            SVM                    0.849494
Name: Accuracy, dtype: float64

In [8]:
# cell 8

import os

PROJECT_ROOT = os.path.abspath("..")

results_folder = os.path.join(PROJECT_ROOT, "results")

os.makedirs(results_folder, exist_ok=True)

results_path = os.path.join(results_folder, "raw_results_adult.csv")

results_df.to_csv(results_path, index=False)

print("Saved successfully at:")
print(results_path)

Saved successfully at:
D:\CSE\IOHE-IndustryOrientedHandsOnExp\Robustness_Project\results\raw_results_adult.csv
